# Gather and Preprocess Valid/Test Sets for Evaluation

## Imports

In [ ]:
import ast
import glob
import gzip
import io
import os
import shutil
import tempfile

# Third-Party Libraries
import tqdm

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import biotite
import biotite.structure

import atomworks
from atomworks.enums import ChainType
from atomworks.ml.utils.token import get_token_starts

from na_eval_utils import (
    read_cluster_ids_text_file,
    load_first_assembly_parsed_and_atom_array,
    load_first_assembly_atom_array,
    chain_num_to_chain_id
)

## Structural Deduplication for Design Dataset

### Setup

In [ ]:
design_all_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/all.csv"
design_train_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/train.csv"
design_valid_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/valid.csv"
design_test_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/test.csv"

legacy_cif_dir = "/home/akubaney/projects/na_mpnn/evaluation/structural_similarity/legacy_cifs"
chain_cifs_directory = "/home/akubaney/projects/na_mpnn/evaluation/structural_similarity/chain_cifs"
us_align_per_chain_directory = "/home/akubaney/projects/na_mpnn/evaluation/structural_similarity/usalign_per_chain"

us_align_overall_output_path = "/home/akubaney/projects/na_mpnn/evaluation/structural_similarity/usalign_output.parquet"

In [ ]:
design_df_all = pd.read_csv(design_all_csv_path)
design_df_train = pd.read_csv(design_train_csv_path)
design_df_valid = pd.read_csv(design_valid_csv_path)
design_df_test = pd.read_csv(design_test_csv_path)

### Break Train/Valid/Test Structures into Per-Chain CIF Files

```
cd ./structural_similarity
sbatch ./save_chain_cifs_array.sh
```

In [ ]:
# Check success that all PDBs in the design dataset are represented.
design_all_pdbs = set(design_df_all.id.unique())

chain_cifs_paths = glob.glob(f"{chain_cifs_directory}/**/*.cif")
chain_cifs_pdbs = set(
    [
        path.split(os.sep)[-1].split(".")[0].split("_")[0] 
        for path in chain_cifs_paths
    ]
)

print(f"Success: {design_all_pdbs == chain_cifs_pdbs}")

### Run US-Align on All Pairs

```
cd ./structural_similarity
sbatch ./run_us_align_array.sh
```

In [ ]:
design_train_and_valid_pdbs = set(design_df_train.id.unique()).union(set(design_df_valid.id.unique()))
train_and_valid_chain_cif_paths = [
    path for path in chain_cifs_paths 
    if path.split(os.sep)[-1].split(".")[0].split("_")[0] in design_train_and_valid_pdbs
]
us_align_output_paths = sorted(glob.glob(f"{us_align_per_chain_directory}/**/*.csv"))

print(f"Success: {len(train_and_valid_chain_cif_paths) == len(us_align_output_paths)}")

In [ ]:
# Remove a few data-dense columns that we don't need for the evaluation.
columns_to_keep = [
    "reference_pdb_id",
    "reference_chain_id",
    "subject_pdb_id",
    "subject_chain_id",
    "loaded_subject_length",
    "loaded_reference_length",
    "aligned_length",
    "sequence_identity (n_identical/n_aligned)",
    "rmsd",
    "tm_score",
    "error_sequence_too_short",
    "error_cannot_parse_file"
]

string_columns = [
    "reference_pdb_id",
    "reference_chain_id",
    "subject_pdb_id",
    "subject_chain_id",
]

writer = None

for us_align_output_path in tqdm.tqdm(us_align_output_paths):
    df = pd.read_csv(
        us_align_output_path,
        usecols=columns_to_keep,
        dtype={
            "loaded_subject_length": float,
            "loaded_reference_length": float,
            "aligned_length": float,
            "sequence_identity (n_identical/n_aligned)": float,
            "rmsd": float,
            "tm_score": float,
            "error_sequence_too_short": bool,
            "error_cannot_parse_file": bool,
        },
        converters={col: (lambda x: x) for col in string_columns},
    )

    table = pa.Table.from_pandas(df, preserve_index = False)

    if writer is None:
        writer = pq.ParquetWriter(us_align_overall_output_path, table.schema)

    writer.write_table(table)

if writer is not None:
    writer.close()

## Imports

## Subsetting Functions

In [ ]:
def get_exclusive_cluster_subset(dataset_df, 
                                 cluster_ids_column_name, 
                                 dataset_cluster_ids):
    """
    Get a subset of the dataset dataframe where the cluster ids in the
    specified column are all in the provided cluster ids for the dataset.
    """
    # Subset the dataframe to only include rows where all cluster ids
    # are in the provided cluster ids.
    dataset_df_subset = dataset_df[
        dataset_df[cluster_ids_column_name].apply(
            lambda cluster_ids_str: all(
                [cluster_id in dataset_cluster_ids for cluster_id in ast.literal_eval(cluster_ids_str)]
            )
        )
    ].copy()

    return dataset_df_subset

def get_length_subset(dataset_df,
                      max_length,
                      min_length):
    """
    Get a subset of the dataset dataframe where the assembly lengths
    are within the specified range.
    """
    valid_length_entries = set()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()

        asmb_lengths_path = row_dict["asmb_lengths_path"]

        # Read the assembly lengths from the file.
        asmb_lengths = np.load(asmb_lengths_path, allow_pickle = True).item()

        for assembly_id in asmb_lengths:
            (macromolecule_L, protein_L, dna_L, rna_L) = asmb_lengths[assembly_id]

            if macromolecule_L <= max_length and macromolecule_L >= min_length:
                valid_length_entries.add(row_dict["id"])
                break
        
    # Subset the dataframe to only include rows where the assembly lengths
    # are valid.
    dataset_df_subset = dataset_df[
        dataset_df["id"].apply(
            lambda id: id in valid_length_entries
        )
    ].copy()

    return dataset_df_subset

def get_ppm_subset(dataset_df):
    """
    Get a subset of the dataset dataframe where the PPM paths are not empty.
    """
    # Subset the dataframe to only include rows where the PPM paths are not empty.
    dataset_df_subset = dataset_df[
        dataset_df["ppm_paths"].apply(
            lambda ppm_paths_str: len(ast.literal_eval(ppm_paths_str)) > 0
        )
    ].copy()

    return dataset_df_subset

def get_entries_in_same_clusters_as_specified_entries(
    dataset_df,
    entry_ids,
    cluster_ids_column_name,
):
    """
    Get a subset of the dataset dataframe where the clusters fall into the
    same clusters as the specified entries.
    """
    entry_cluster_ids = set()
    for entry_id in entry_ids:
        row_dict = dataset_df[dataset_df["id"] == entry_id].iloc[0].to_dict()
        cluster_ids = ast.literal_eval(row_dict[cluster_ids_column_name])
        entry_cluster_ids.update(cluster_ids)

    dataset_df_subset = dataset_df[
        dataset_df[cluster_ids_column_name].apply(
            lambda cluster_ids_str: all(
                [cluster_id in entry_cluster_ids for cluster_id in ast.literal_eval(cluster_ids_str)]
            )
        )
    ].copy()

    return dataset_df_subset

In [ ]:
def get_nucleic_acid_length_subset(
    dataset_df,
    max_length,
    min_length
):
    """
    Get a subset of the dataset dataframe where the assembly lengths
    are within the specified range.
    """
    valid_length_entries = set()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()

        asmb_lengths_path = row_dict["asmb_lengths_path"]

        # Read the assembly lengths from the file.
        asmb_lengths = np.load(asmb_lengths_path, allow_pickle = True).item()

        for assembly_id in asmb_lengths:
            (macromolecule_L, protein_L, dna_L, rna_L) = asmb_lengths[assembly_id]

            na_L = dna_L + rna_L

            if na_L <= max_length and na_L >= min_length:
                valid_length_entries.add(row_dict["id"])
                break
        
    # Subset the dataframe to only include rows where the assembly lengths
    # are valid.
    dataset_df_subset = dataset_df[
        dataset_df["id"].apply(
            lambda id: id in valid_length_entries
        )
    ].copy()

    return dataset_df_subset

def get_chain_resolution_info(atom_array):
    """
    Get per-chain resolution information for protein and nucleic acid chains.

    Leading and trailing unresolved residues are allowed. Internal unresolved
    residues are not allowed once the resolved segment begins.

    For protein chains, also records whether the resolved middle segment
    contains only canonical amino acids, as recovered from the atom array.

    Returns:
        chain_resolution_info (list of dict): One dict per protein or nucleic
            acid chain, containing whether each token is resolved, whether the
            chain has an internal break, whether the resolved middle region of a
            protein chain is canonical, and the atom bounds corresponding to
            the first resolved token through the last resolved token.
    """
    protein_resname_to_restype = {
        "ALA": "A",
        "ARG": "R",
        "ASN": "N",
        "ASP": "D",
        "CYS": "C",
        "GLN": "Q",
        "GLU": "E",
        "GLY": "G",
        "HIS": "H",
        "ILE": "I",
        "LEU": "L",
        "LYS": "K",
        "MET": "M",
        "PHE": "F",
        "PRO": "P",
        "SER": "S",
        "THR": "T",
        "TRP": "W",
        "TYR": "Y",
        "VAL": "V",
    }

    chain_resolution_info = []

    # Use chain_iid to account for symmetry.
    ordered_chain_iids = list(dict.fromkeys(atom_array.chain_iid.tolist()))

    # Check each chain for internal breaks.
    for chain_iid in ordered_chain_iids:
        # Grab the atom array for the chain.
        chain_mask = (atom_array.chain_iid == chain_iid)
        chain_atom_array = atom_array[chain_mask]

        if len(chain_atom_array) == 0:
            raise ValueError(f"chain_iid {chain_iid} has no atoms.")

        # Check that the chain type is consistent across the chain.
        chain_types = np.unique(chain_atom_array.chain_type)
        if len(chain_types) != 1:
            raise ValueError(
                f"Chain {chain_atom_array.chain_id[0]} "
                f"(chain_iid={chain_iid}) has multiple chain types: "
                f"{chain_types.tolist()}"
            )

        chain_type = chain_types[0]

        # Skip non-protein / non-nucleic acid chains.
        if chain_type not in (
            ChainType.POLYPEPTIDE_L,
            ChainType.DNA,
            ChainType.RNA,
            ChainType.DNA_RNA_HYBRID
        ):
            continue

        # Get the atoms at the token starts and ends.
        token_starts = get_token_starts(chain_atom_array)
        if len(token_starts) == 0:
            raise ValueError(
                f"Chain {chain_atom_array.chain_id[0]} "
                f"(chain_iid={chain_iid}) has no tokens."
            )
        token_ends = list(token_starts[1:]) + [len(chain_atom_array)]

        token_is_resolved = []
        token_protein_restypes = []
        for token_start, token_end in zip(token_starts, token_ends):
            token_atom_array = chain_atom_array[token_start:token_end]

            token_res_names = np.unique(token_atom_array.res_name)
            if len(token_res_names) != 1:
                raise ValueError(
                    f"Chain {chain_atom_array.chain_id[0]} "
                    f"(chain_iid={chain_iid}) token has multiple residue names: "
                    f"{token_res_names.tolist()}"
                )
            token_res_name = token_res_names[0]

            if chain_type == ChainType.POLYPEPTIDE_L:
                required_backbone_atoms = ["N", "CA", "C", "O"]
                token_protein_restype = protein_resname_to_restype.get(
                    token_res_name,
                    "X"
                )
            elif chain_type == ChainType.RNA:
                required_backbone_atoms = [
                    "OP1", "OP2", "P", "O5'", "C5'", "C4'", "O4'",
                    "C3'", "O3'", "C2'", "O2'", "C1'"
                ]
                token_protein_restype = None
            else:
                required_backbone_atoms = [
                    "OP1", "OP2", "P", "O5'", "C5'", "C4'", "O4'",
                    "C3'", "O3'", "C2'", "C1'"
                ]
                token_protein_restype = None

            resolved = True
            # Check each required backbone atom.
            for atom_name in required_backbone_atoms:
                # Check atom existence.
                atom_mask = (token_atom_array.atom_name == atom_name)
                if not np.any(atom_mask):
                    resolved = False
                    break

                # Check atom coordinates (not nan).
                atom_coords = token_atom_array.coord[atom_mask]
                has_finite_coords = np.any(
                    np.all(np.isfinite(atom_coords), axis = 1)
                )
                if not has_finite_coords:
                    resolved = False
                    break

            # Mark the token as resolved.
            token_is_resolved.append(resolved)
            token_protein_restypes.append(token_protein_restype)

        # Convert to array.
        token_is_resolved = np.array(token_is_resolved, dtype = bool)

        # Check for internal breaks.
        resolved_token_indices = np.where(token_is_resolved)[0]
        if len(resolved_token_indices) == 0:
            has_internal_chain_break = True
            first_resolved_idx = None
            last_resolved_idx = None
            keep_atom_start_idx = None
            keep_atom_end_idx = None
            protein_middle_sequence = None
            protein_middle_all_canonical = False
        else:
            first_resolved_idx = resolved_token_indices[0]
            last_resolved_idx = resolved_token_indices[-1]

            has_internal_chain_break = not np.all(
                token_is_resolved[first_resolved_idx : last_resolved_idx + 1]
            )

            # Keep from the first resolved token through the last resolved
            # token, dropping unresolved residues at the chain ends.
            keep_atom_start_idx = token_starts[first_resolved_idx]
            keep_atom_end_idx = token_ends[last_resolved_idx]

            if chain_type == ChainType.POLYPEPTIDE_L:
                protein_middle_sequence = "".join(
                    token_protein_restypes[
                        first_resolved_idx : last_resolved_idx + 1
                    ]
                )
                protein_middle_all_canonical = (
                    "X" not in protein_middle_sequence
                )
            else:
                protein_middle_sequence = None
                protein_middle_all_canonical = True

        chain_resolution_info.append(
            {
                "chain_iid": chain_iid,
                "chain_id": chain_atom_array.chain_id[0],
                "chain_type": chain_type,
                "token_starts": token_starts,
                "token_ends": token_ends,
                "token_is_resolved": token_is_resolved,
                "has_internal_chain_break": has_internal_chain_break,
                "first_resolved_idx": first_resolved_idx,
                "last_resolved_idx": last_resolved_idx,
                "keep_atom_start_idx": keep_atom_start_idx,
                "keep_atom_end_idx": keep_atom_end_idx,
                "protein_middle_sequence": protein_middle_sequence,
                "protein_middle_all_canonical": protein_middle_all_canonical,
            }
        )

    return chain_resolution_info

def get_low_protein_subset(dataset_df):
    """
    Get a subset of the dataset dataframe where the assembly has no more than
    2 protein chains and, if there are 2 protein chains, the resolved middle
    chunks that would actually be saved are the same sequence.
    """
    valid_entries = set()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()
        structure_path = row_dict["structure_path"]

        # Check for path existence.
        if os.path.exists(structure_path):
            structure_path = structure_path
        else:
            structure_path = legacy_cif_dir + f"/{row_dict['id']}.cif.gz"

        # Load the structure.
        atom_array = load_first_assembly_atom_array(
            structure_path
        )

        # Get per-chain resolution information so that protein sequences are
        # taken from the resolved middle chunk that would actually be saved.
        chain_resolution_info = get_chain_resolution_info(
            atom_array
        )

        # Grab protein sequences from each kept protein chain.
        protein_sequences = []
        for chain_info in chain_resolution_info:
            if chain_info["chain_type"] != ChainType.POLYPEPTIDE_L:
                continue

            protein_sequences.append(
                chain_info["protein_middle_sequence"]
            )

        # Skip entries with more than 2 protein chains or 2 protein chains with
        # different resolved middle sequences.
        if len(protein_sequences) > 2:
            continue

        if len(protein_sequences) == 2:
            if protein_sequences[0] != protein_sequences[1]:
                continue

        valid_entries.add(row_dict["id"])

    # Subset the dataframe to only include valid entries.
    dataset_df_subset = dataset_df[
        dataset_df["id"].apply(
            lambda id: id in valid_entries
        )
    ].copy()

    return dataset_df_subset

################################################################################
# Chain resolution helpers
################################################################################
def get_chain_resolution_info(atom_array):
    """
    Get per-chain resolution information for protein and nucleic acid chains.

    Leading and trailing unresolved residues are allowed. Internal unresolved
    residues are not allowed once the resolved segment begins.

    For protein chains, also records whether the resolved middle segment
    contains only canonical amino acids, as recovered from the atom array.

    Returns:
        chain_resolution_info (list of dict): One dict per protein or nucleic
            acid chain, containing whether each token is resolved, whether the
            chain has an internal break, whether the resolved middle region of a
            protein chain is canonical, and the atom bounds corresponding to
            the first resolved token through the last resolved token.
    """
    protein_resname_to_restype = {
        "ALA": "A",
        "ARG": "R",
        "ASN": "N",
        "ASP": "D",
        "CYS": "C",
        "GLN": "Q",
        "GLU": "E",
        "GLY": "G",
        "HIS": "H",
        "ILE": "I",
        "LEU": "L",
        "LYS": "K",
        "MET": "M",
        "PHE": "F",
        "PRO": "P",
        "SER": "S",
        "THR": "T",
        "TRP": "W",
        "TYR": "Y",
        "VAL": "V",
    }

    chain_resolution_info = []

    # Use chain_iid to account for symmetry.
    ordered_chain_iids = list(dict.fromkeys(atom_array.chain_iid.tolist()))

    # Check each chain for internal breaks.
    for chain_iid in ordered_chain_iids:
        # Grab the atom array for the chain.
        chain_mask = (atom_array.chain_iid == chain_iid)
        chain_atom_array = atom_array[chain_mask]

        if len(chain_atom_array) == 0:
            raise ValueError(f"chain_iid {chain_iid} has no atoms.")

        # Check that the chain type is consistent across the chain.
        chain_types = np.unique(chain_atom_array.chain_type)
        if len(chain_types) != 1:
            raise ValueError(
                f"Chain {chain_atom_array.chain_id[0]} "
                f"(chain_iid={chain_iid}) has multiple chain types: "
                f"{chain_types.tolist()}"
            )

        chain_type = chain_types[0]

        # Skip non-protein / non-nucleic acid chains.
        if chain_type not in (
            ChainType.POLYPEPTIDE_L,
            ChainType.DNA,
            ChainType.RNA,
            ChainType.DNA_RNA_HYBRID
        ):
            continue

        # Get the atoms at the token starts and ends.
        token_starts = get_token_starts(chain_atom_array)
        if len(token_starts) == 0:
            raise ValueError(
                f"Chain {chain_atom_array.chain_id[0]} "
                f"(chain_iid={chain_iid}) has no tokens."
            )
        token_ends = list(token_starts[1:]) + [len(chain_atom_array)]

        token_is_resolved = []
        token_protein_restypes = []
        for token_start, token_end in zip(token_starts, token_ends):
            token_atom_array = chain_atom_array[token_start:token_end]

            token_res_names = np.unique(token_atom_array.res_name)
            if len(token_res_names) != 1:
                raise ValueError(
                    f"Chain {chain_atom_array.chain_id[0]} "
                    f"(chain_iid={chain_iid}) token has multiple residue names: "
                    f"{token_res_names.tolist()}"
                )
            token_res_name = token_res_names[0]

            if chain_type == ChainType.POLYPEPTIDE_L:
                required_backbone_atoms = ["N", "CA", "C", "O"]
                token_protein_restype = protein_resname_to_restype.get(
                    token_res_name,
                    "X"
                )
            elif chain_type == ChainType.RNA:
                required_backbone_atoms = [
                    "OP1", "OP2", "P", "O5'", "C5'", "C4'", "O4'",
                    "C3'", "O3'", "C2'", "O2'", "C1'"
                ]
                token_protein_restype = None
            else:
                required_backbone_atoms = [
                    "OP1", "OP2", "P", "O5'", "C5'", "C4'", "O4'",
                    "C3'", "O3'", "C2'", "C1'"
                ]
                token_protein_restype = None

            resolved = True
            # Check each required backbone atom.
            for atom_name in required_backbone_atoms:
                # Check atom existence.
                atom_mask = (token_atom_array.atom_name == atom_name)
                if not np.any(atom_mask):
                    resolved = False
                    break

                # Check atom coordinates (not nan).
                atom_coords = token_atom_array.coord[atom_mask]
                has_finite_coords = np.any(
                    np.all(np.isfinite(atom_coords), axis = 1)
                )
                if not has_finite_coords:
                    resolved = False
                    break

            # Mark the token as resolved.
            token_is_resolved.append(resolved)
            token_protein_restypes.append(token_protein_restype)

        # Convert to array.
        token_is_resolved = np.array(token_is_resolved, dtype = bool)

        # Check for internal breaks.
        resolved_token_indices = np.where(token_is_resolved)[0]
        if len(resolved_token_indices) == 0:
            has_internal_chain_break = True
            first_resolved_idx = None
            last_resolved_idx = None
            keep_atom_start_idx = None
            keep_atom_end_idx = None
            protein_middle_sequence = None
            protein_middle_all_canonical = False
        else:
            first_resolved_idx = resolved_token_indices[0]
            last_resolved_idx = resolved_token_indices[-1]

            has_internal_chain_break = not np.all(
                token_is_resolved[first_resolved_idx : last_resolved_idx + 1]
            )

            # Keep from the first resolved token through the last resolved
            # token, dropping unresolved residues at the chain ends.
            keep_atom_start_idx = token_starts[first_resolved_idx]
            keep_atom_end_idx = token_ends[last_resolved_idx]

            if chain_type == ChainType.POLYPEPTIDE_L:
                protein_middle_sequence = "".join(
                    token_protein_restypes[
                        first_resolved_idx : last_resolved_idx + 1
                    ]
                )
                protein_middle_all_canonical = (
                    "X" not in protein_middle_sequence
                )
            else:
                protein_middle_sequence = None
                protein_middle_all_canonical = True

        chain_resolution_info.append(
            {
                "chain_iid": chain_iid,
                "chain_id": chain_atom_array.chain_id[0],
                "chain_type": chain_type,
                "token_starts": token_starts,
                "token_ends": token_ends,
                "token_is_resolved": token_is_resolved,
                "has_internal_chain_break": has_internal_chain_break,
                "first_resolved_idx": first_resolved_idx,
                "last_resolved_idx": last_resolved_idx,
                "keep_atom_start_idx": keep_atom_start_idx,
                "keep_atom_end_idx": keep_atom_end_idx,
                "protein_middle_sequence": protein_middle_sequence,
                "protein_middle_all_canonical": protein_middle_all_canonical,
            }
        )

    return chain_resolution_info

def get_no_chain_breaks_subset(dataset_df):
    """
    Get a subset of the dataset dataframe where every protein and nucleic acid
    chain contains a contiguous resolved segment with no internal break.

    Leading and trailing unresolved residues are allowed. Internal unresolved
    residues are not allowed once the resolved segment begins.

    Also require that every resolved protein middle segment contains only
    canonical amino acids, as recovered from the atom array.
    """
    valid_entries = set()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()
        structure_path = row_dict["structure_path"]

        # Check for path existence.
        if os.path.exists(structure_path):
            structure_path = structure_path
        else:
            structure_path = legacy_cif_dir + f"/{row_dict['id']}.cif.gz"
        
        # Load the structure.
        atom_array = load_first_assembly_atom_array(
            structure_path
        )

        chain_resolution_info = get_chain_resolution_info(
            atom_array
        )

        has_invalid_chain = False
        # Check each chain for internal breaks or non-canonical protein
        # residues in the resolved middle segment.
        for chain_info in chain_resolution_info:
            if chain_info["has_internal_chain_break"]:
                has_invalid_chain = True
                break

            if (
                chain_info["chain_type"] == ChainType.POLYPEPTIDE_L and
                not chain_info["protein_middle_all_canonical"]
            ):
                has_invalid_chain = True
                break
        
        # If no invalid chains were found, add the entry to the valid entries.
        if not has_invalid_chain:
            valid_entries.add(row_dict["id"])

    # Subset to only include valid entries.
    dataset_df_subset = dataset_df[
        dataset_df["id"].apply(
            lambda id: id in valid_entries
        )
    ].copy()

    return dataset_df_subset

def get_nucleic_acid_structure_deduplication_subset(
    dataset_df,
    us_align_parquet_path,
    tm_score_threshold = 0.5,
    coverage_threshold = 0.8,
):
    """
    Get a subset of the dataset dataframe where no nucleic acid chain has a
    structural hit above the specified thresholds.
    """
    us_align_df = pd.read_parquet(
        us_align_parquet_path,
        columns=[
            "reference_pdb_id",
            "loaded_reference_length",
            "aligned_length",
            "tm_score",
            "error_sequence_too_short",
            "error_cannot_parse_file",
        ]
    )

    # Filter out alignments with errors.
    us_align_df = us_align_df[
        ~us_align_df["error_sequence_too_short"]
    ]
    us_align_df = us_align_df[
        ~us_align_df["error_cannot_parse_file"]
    ]

    # Compute coverage and filter out alignments with low coverage.
    us_align_df["coverage"] = (
        us_align_df["aligned_length"] /
        us_align_df["loaded_reference_length"]
    )
    us_align_df = us_align_df[
        us_align_df["coverage"] >= coverage_threshold
    ]

    # For each reference PDB ID, get the maximum TM-score across all alignments.
    max_tm_score_by_reference_pdb_id = us_align_df.groupby(
        "reference_pdb_id"
    )["tm_score"].max()

    # Deduplicate by filtering out entries where the reference PDB ID has a
    # maximum TM-score above the threshold.
    deduplicated_reference_pdb_ids = set(
        max_tm_score_by_reference_pdb_id[
            max_tm_score_by_reference_pdb_id >= tm_score_threshold
        ].index
    )
    dataset_df_subset = dataset_df[
        ~dataset_df["id"].isin(deduplicated_reference_pdb_ids)
    ].copy()

    return dataset_df_subset

def get_one_per_nucleic_acid_chain_cluster_subset(dataset_df):
    """
    Get a subset of the dataset dataframe where each entry contributes at least
    one nucleic acid chain cluster that has not been seen previously.

    Selection is made deterministically by sorting by id first.
    """
    # Sort the dataframe by id to ensure deterministic selection.
    dataset_df = dataset_df.sort_values(
        "id",
        kind="mergesort"
    ).reset_index(drop=True)

    # Note which cluster ids have been seen as we iterate through the
    # dataframe, and only keep entries that contribute at least one new
    # cluster id.
    seen_cluster_ids = set()
    valid_entries = set()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()

        cluster_ids_value = row_dict["nucleic_acid_chain_cluster_ids"]
        if isinstance(cluster_ids_value, str):
            cluster_ids = ast.literal_eval(cluster_ids_value)
        else:
            cluster_ids = list(cluster_ids_value)

        if any([cluster_id not in seen_cluster_ids for cluster_id in cluster_ids]):
            valid_entries.add(row_dict["id"])
            seen_cluster_ids.update(cluster_ids)

    dataset_df_subset = dataset_df[
        dataset_df["id"].apply(
            lambda id: id in valid_entries
        )
    ].copy()

    return dataset_df_subset

In [ ]:
def convert_cif_to_pdb(dataset_df, 
                       pdb_output_directory,  
                       use_rna_solo = False,
                       pdb_id_to_rna_solo_paths = None):
    """
    Convert the CIF files in the dataset dataframe to PDB files.
    """
    if use_rna_solo:
        assert(pdb_id_to_rna_solo_paths is not None)
    
    os.makedirs(pdb_output_directory)
    
    dataset_df = dataset_df.copy()
    successfully_converted_ids = set()
    pdb_id_to_structure_path = dict()
    pdb_id_to_original_structure_path = dict()
    pdb_id_to_copied_structure_path = dict()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()

        # Fetch the original structure path.
        original_structure_path = row_dict["structure_path"]
        
        # Compute the destination structure path.
        destination_structure_path = os.path.join(
            pdb_output_directory,
            row_dict["id"] + ".pdb"
        )

        if use_rna_solo:
            # Use the first RNA-Solo path if it exists.
            rna_solo_paths = pdb_id_to_rna_solo_paths.get(row_dict["id"], [])
            if len(rna_solo_paths) == 0:
                continue
            structure_path_to_copy = rna_solo_paths[0]
        else:
            structure_path_to_copy = original_structure_path
        
        # Use a temp file to handle gzipped files.
        try:
            if structure_path_to_copy.endswith(".gz"):
                structure_ext = os.path.splitext(structure_path_to_copy[:-3])[1]
                
                with (gzip.open(structure_path_to_copy, "rb") as f_in,
                    tempfile.NamedTemporaryFile(suffix = structure_ext) as tmp):
                    tmp.write(f_in.read())
                    tmp.flush()
                    atom_array = biotite.structure.io.load_structure(tmp.name)
                    tmp.close()
            else:
                atom_array = biotite.structure.io.load_structure(structure_path_to_copy)
            
            biotite.structure.io.save_structure(destination_structure_path, atom_array)
        except:
            continue
            
        # Add the id to the set of successfully converted ids.
        successfully_converted_ids.add(row_dict["id"])

        # Replace the destination structure path, original structure path,
        # and copied structure path in the dataframe.
        pdb_id_to_structure_path[row_dict["id"]] = destination_structure_path
        pdb_id_to_original_structure_path[row_dict["id"]] = original_structure_path
        pdb_id_to_copied_structure_path[row_dict["id"]] = structure_path_to_copy
    
    # Create a new dataframe with the successfully converted ids.
    dataset_subset_df = dataset_df[
        dataset_df["id"].apply(
            lambda id: id in successfully_converted_ids
        )
    ].copy()

    # Replace the structure path in the new dataframe with the destination
    # structure path.
    dataset_subset_df["structure_path"] = dataset_subset_df["id"].apply(
        lambda id: pdb_id_to_structure_path[id]
    )
    dataset_subset_df["original_structure_path"] = dataset_subset_df["id"].apply(
        lambda id: pdb_id_to_original_structure_path[id]
    )
    dataset_subset_df["copied_structure_path"] = dataset_subset_df["id"].apply(
        lambda id: pdb_id_to_copied_structure_path[id]
    )

    return dataset_subset_df

In [ ]:
def convert_cif_to_pdb_robust(dataset_df, pdb_output_directory):
    """
    Convert the CIF files in the dataset dataframe to PDB files using the first
    assembly, renaming chains by chain_iid and keeping only protein or nucleic
    acid chains.

    Protein chains are moved to the end before assigning new chain IDs.

    Protein and nucleic acid chains are trimmed from the first resolved token to
    the last resolved token, dropping unresolved residues at the chain ends.

    Raises an error if any chain contains more than one chain type.
    """
    os.makedirs(pdb_output_directory, exist_ok = True)

    dataset_df = dataset_df.copy()
    successfully_converted_ids = set()
    pdb_id_to_structure_path = dict()
    pdb_id_to_original_structure_path = dict()
    pdb_id_to_copied_structure_path = dict()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()

        # Fetch the original structure path.
        original_structure_path = row_dict["structure_path"]

        # Check for path existence.
        if os.path.exists(original_structure_path):
            structure_path = original_structure_path
        else:
            structure_path = legacy_cif_dir + f"/{row_dict['id']}.cif.gz"

        # Compute the destination structure path.
        destination_structure_path = os.path.join(
            pdb_output_directory,
            row_dict["id"] + ".pdb"
        )

        try:
            # Load the first assembly.
            atom_array = load_first_assembly_atom_array(
                structure_path
            )

            chain_resolution_info = get_chain_resolution_info(
                atom_array
            )
            chain_iid_to_resolution_info = {
                chain_iid_info["chain_iid"]: chain_iid_info
                for chain_iid_info in chain_resolution_info
            }

            # Keep only chains that are entirely protein or nucleic acid. Raise
            # an error if any chain contains more than one chain type.
            input_order_chain_iids = list(
                dict.fromkeys(atom_array.chain_iid.tolist())
            )
            protein_chain_iids = []
            non_protein_chain_iids = []
            chain_iid_to_kept_atom_indices = dict()
            for chain_iid in input_order_chain_iids:
                chain_mask = (atom_array.chain_iid == chain_iid)
                chain_atom_array = atom_array[chain_mask]

                if len(chain_atom_array) == 0:
                    raise ValueError(f"chain_iid {chain_iid} has no atoms.")

                # Check that the chain type is consistent across the chain.
                chain_types = np.unique(chain_atom_array.chain_type)
                if len(chain_types) != 1:
                    raise ValueError(
                        f"Chain {chain_atom_array.chain_id[0]} "
                        f"(chain_iid={chain_iid}) in {row_dict['id']} has "
                        f"multiple chain types: {chain_types.tolist()}"
                    )

                # Keep the chain if it is protein or nucleic acid.
                chain_type = chain_types[0]
                if chain_type in (
                    ChainType.POLYPEPTIDE_L,
                    ChainType.DNA,
                    ChainType.RNA,
                    ChainType.DNA_RNA_HYBRID
                ):
                    chain_iid_info = chain_iid_to_resolution_info[
                        chain_iid
                    ]

                    if chain_type == ChainType.POLYPEPTIDE_L:
                        if not chain_iid_info["protein_middle_all_canonical"]:
                            raise ValueError(
                                f"Non-canonical protein residues found in "
                                f"{row_dict['id']} chain "
                                f"{chain_atom_array.chain_id[0]} "
                                f"(chain_iid={chain_iid}): "
                                f"{chain_iid_info['protein_middle_sequence']}"
                            )
                        protein_chain_iids.append(chain_iid)
                    else:
                        non_protein_chain_iids.append(chain_iid)

                    # Subset from the first resolved token to the last
                    # resolved token, dropping unresolved residues at the
                    # chain ends.
                    chain_atom_indices = np.where(
                        atom_array.chain_iid == chain_iid
                    )[0]
                    kept_atom_indices = chain_atom_indices[
                        chain_iid_info["keep_atom_start_idx"] :
                        chain_iid_info["keep_atom_end_idx"]
                    ]

                    chain_iid_to_kept_atom_indices[chain_iid] = kept_atom_indices

            ordered_chain_iids = non_protein_chain_iids + protein_chain_iids

            if len(ordered_chain_iids) == 0:
                raise ValueError(
                    f"No protein or nucleic acid chains found in {row_dict['id']}."
                )

            # Reorder the atom array so non-protein chains come first and
            # protein chains come last, preserving the original atom order
            # within each chain.
            ordered_atom_indices = np.concatenate([
                chain_iid_to_kept_atom_indices[chain_iid]
                for chain_iid in ordered_chain_iids
            ])
            atom_array = atom_array[ordered_atom_indices]

            if len(atom_array) == 0:
                raise ValueError(
                    f"No atoms left after filtering for protein and nucleic acid "
                    f"chains in {row_dict['id']}."
                )

            # Rename chains purely by chain_iid in the new order.
            chain_iid_to_new_chain_id = {
                chain_iid: chain_num_to_chain_id(i)
                for i, chain_iid in enumerate(ordered_chain_iids)
            }

            atom_array = atom_array.copy()
            atom_array.chain_id = atom_array.chain_id.astype("<U8")
            for chain_iid, new_chain_id in chain_iid_to_new_chain_id.items():
                chain_mask = (atom_array.chain_iid == chain_iid)
                atom_array.chain_id[chain_mask] = new_chain_id

            # Round-trip through CIF text with NaN coordinates excluded before
            # writing the PDB.
            cif_string = atomworks.io.utils.io_utils.to_cif_string(
                atom_array,
                include_nan_coords = False,
                include_entity_poly = False
            )
            atom_array = load_first_assembly_atom_array(
                io.StringIO(cif_string),
                add_missing_atoms = False
            )
            pdb_string = atomworks.io.utils.io_utils.to_pdb_string(atom_array)

            with open(destination_structure_path, "wt") as f:
                f.write(pdb_string)
        except Exception as e:
            print(f"Error occurred while processing {row_dict['id']}: {e}")
            continue

        # Add the id to the set of successfully converted ids.
        successfully_converted_ids.add(row_dict["id"])

        # Replace the destination structure path, original structure path,
        # and copied structure path in the dataframe.
        pdb_id_to_structure_path[row_dict["id"]] = destination_structure_path
        pdb_id_to_original_structure_path[row_dict["id"]] = original_structure_path
        pdb_id_to_copied_structure_path[row_dict["id"]] = structure_path

    # Create a new dataframe with the successfully converted ids.
    dataset_subset_df = dataset_df[
        dataset_df["id"].apply(
            lambda id: id in successfully_converted_ids
        )
    ].copy()

    # Replace the structure path in the new dataframe with the destination
    # structure path.
    dataset_subset_df["structure_path"] = dataset_subset_df["id"].apply(
        lambda id: pdb_id_to_structure_path[id]
    )
    dataset_subset_df["original_structure_path"] = dataset_subset_df["id"].apply(
        lambda id: pdb_id_to_original_structure_path[id]
    )
    dataset_subset_df["copied_structure_path"] = dataset_subset_df["id"].apply(
        lambda id: pdb_id_to_copied_structure_path[id]
    )

    return dataset_subset_df

In [ ]:
def get_polymer_type_statistics(dataset_df):
    """
    Print the number of different types of entries in the dataset.
    The types of entries are:
    - DNA
    - RNA
    - DNA/RNA hybrid
    - Protein/DNA
    - Protein/RNA
    - Protein/DNA/RNA hybrid
    """
    dna_entries = set()
    rna_entries = set()
    dna_rna_hybrid_entries = set()
    protein_dna_entries = set()
    protein_rna_entries = set()
    protein_dna_rna_hybrid_entries = set()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()

        nucleic_acid_chain_cluster_ids_chain_types = ast.literal_eval(
            row_dict["nucleic_acid_chain_cluster_ids_chain_types"]
        )
        protein_chain_cluster_ids_chain_types = ast.literal_eval(
            row_dict["protein_chain_cluster_ids_chain_types"]
        )

        has_protein = len(protein_chain_cluster_ids_chain_types) > 0
        has_dna = "polydeoxyribonucleotide" in nucleic_acid_chain_cluster_ids_chain_types
        has_rna = "polyribonucleotide" in nucleic_acid_chain_cluster_ids_chain_types
        has_dna_rna_hybrid = "polydeoxyribonucleotide/polyribonucleotide hybrid" in nucleic_acid_chain_cluster_ids_chain_types

        if has_protein and has_dna and not has_rna and not has_dna_rna_hybrid:
            protein_dna_entries.add(row_dict["id"])
        elif has_protein and has_rna and not has_dna and not has_dna_rna_hybrid:
            protein_rna_entries.add(row_dict["id"])
        elif has_protein and has_dna_rna_hybrid and not has_dna and not has_rna:
            protein_dna_rna_hybrid_entries.add(row_dict["id"])
        elif has_dna and not has_protein and not has_rna and not has_dna_rna_hybrid:
            dna_entries.add(row_dict["id"])
        elif has_rna and not has_protein and not has_dna and not has_dna_rna_hybrid:
            rna_entries.add(row_dict["id"])
        elif has_dna_rna_hybrid and not has_protein and not has_dna and not has_rna:
            dna_rna_hybrid_entries.add(row_dict["id"])

    print("Number of total entries:", len(dataset_df))
    print("Number of DNA entries:", len(dna_entries))
    print("Number of RNA entries:", len(rna_entries))
    print("Number of DNA/RNA hybrid entries:", len(dna_rna_hybrid_entries))
    print("Number of protein/DNA entries:", len(protein_dna_entries))
    print("Number of protein/RNA entries:", len(protein_rna_entries))
    print("Number of protein/DNA/RNA hybrid entries:", len(protein_dna_rna_hybrid_entries))

def get_ppm_statistics(dataset_df):
    """
    Print the number of different types of entries in the dataset.
    The types of entries are:
    - PPM
    - PPM from crystal
    - PPM from distillation
    """
    # Count the number of different types of entries.
    ppm_entries = set()
    ppm_from_crystal_entries = set()
    ppm_from_distillation_entries = set()
    for i in range(len(dataset_df)):
        row_dict = dataset_df.iloc[i].to_dict()

        ppm_paths = ast.literal_eval(
            row_dict["ppm_paths"]
        )

        has_ppm = len(ppm_paths) > 0
        ppm_from_crystal = row_dict["dataset_name"] == "rcsb_cif_na"
        ppm_from_distillation = (row_dict["dataset_name"] == "rf2na_distillation_cis_bp") or (row_dict["dataset_name"] == "rf2na_distillation_transfac")

        if has_ppm:
            ppm_entries.add(row_dict["id"])
            if ppm_from_crystal:
                ppm_from_crystal_entries.add(row_dict["id"])
            elif ppm_from_distillation:
                ppm_from_distillation_entries.add(row_dict["id"])

    print("Number of total entries:", len(dataset_df))
    print("Number of PPM entries:", len(ppm_entries))
    print("Number of PPM from crystal entries:", len(ppm_from_crystal_entries))
    print("Number of PPM from distillation entries:", len(ppm_from_distillation_entries))

## Create the PDB Output Directory and the CSV Output Directory

In [ ]:
pdb_output_directory = os.path.abspath("./evaluation_pdbs")
csv_output_directory = os.path.abspath("./evaluation_csvs")

In [ ]:
if os.path.exists(pdb_output_directory):
    shutil.rmtree(pdb_output_directory)
os.makedirs(pdb_output_directory)

if os.path.exists(csv_output_directory):
    shutil.rmtree(csv_output_directory)
os.makedirs(csv_output_directory)

In [ ]:
design_valid_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/valid.csv"
design_valid_cluster_ids_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/valid_nucleic_acid_chain_cluster_ids.txt"

design_test_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/test.csv"
design_test_cluster_ids_path = "/home/akubaney/projects/na_mpnn/data/datasets/design_dataset_v2/test_nucleic_acid_chain_cluster_ids.txt"

### Validation Set

In [ ]:
design_valid_pdbs_output_directory = os.path.join(pdb_output_directory, "design_valid")
design_valid_csv_output_path = os.path.join(csv_output_directory, "design_valid.csv")

design_valid_df = pd.read_csv(design_valid_csv_path)
design_valid_cluster_ids = read_cluster_ids_text_file(design_valid_cluster_ids_path)
print("Original valid dataset size:", len(design_valid_df))

design_valid_df = get_exclusive_cluster_subset(
    design_valid_df, 
    "nucleic_acid_chain_cluster_ids", 
    design_valid_cluster_ids
)
print("Valid dataset size after exclusive cluster subset:", len(design_valid_df))

design_valid_df = get_length_subset(
    design_valid_df,
    max_length = 1000,
    min_length = 20
)
print("Valid dataset size after length subset:", len(design_valid_df))

design_valid_df = convert_cif_to_pdb(
    design_valid_df, 
    design_valid_pdbs_output_directory, 
    use_rna_solo = False,
    pdb_id_to_rna_solo_paths = None
)
print("Valid dataset size after CIF to PDB conversion:", len(design_valid_df))

get_polymer_type_statistics(design_valid_df)

design_valid_df.to_csv(design_valid_csv_output_path, index = False)
design_valid_df

### Test Set

In [ ]:
design_test_pdbs_output_directory = os.path.join(pdb_output_directory, "design_test")
design_test_csv_output_path = os.path.join(csv_output_directory, "design_test.csv")

design_test_df = pd.read_csv(design_test_csv_path)
design_test_cluster_ids = read_cluster_ids_text_file(design_test_cluster_ids_path)
print("Original test dataset size:", len(design_test_df))

design_test_df = get_exclusive_cluster_subset(
    design_test_df, 
    "nucleic_acid_chain_cluster_ids", 
    design_test_cluster_ids
)
print("Test dataset size after exclusive cluster subset:", len(design_test_df))

design_test_df = get_length_subset(
    design_test_df,
    max_length = 500,
    min_length = 20
)
print("Test dataset size after length subset:", len(design_test_df))

design_test_df = get_nucleic_acid_length_subset(
    design_test_df,
    max_length = 500,
    min_length = 10
)
print("Test dataset size after nucleic acid length subset:", len(design_test_df))

design_test_df = get_low_protein_subset(design_test_df)
print("Test dataset size after low protein subset:", len(design_test_df))

design_test_df = get_no_chain_breaks_subset(design_test_df)
print("Test dataset size after no chain breaks subset:", len(design_test_df))

design_test_df = get_nucleic_acid_structure_deduplication_subset(design_test_df, us_align_parquet_path = us_align_overall_output_path)
print("Test dataset size after nucleic acid structure deduplication subset:", len(design_test_df))

design_test_df = get_one_per_nucleic_acid_chain_cluster_subset(design_test_df)
print("Test dataset size after one per nucleic acid chain cluster subset:", len(design_test_df))

design_test_df = convert_cif_to_pdb_robust(
    design_test_df,
    design_test_pdbs_output_directory
)
print("Test dataset size after CIF to PDB conversion:", len(design_test_df))

get_polymer_type_statistics(design_test_df)

design_test_df.to_csv(design_test_csv_output_path, index = False)
design_test_df

## Specificity Dataset

In [ ]:
specificity_valid_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/specificity_dataset_v2/valid.csv"
specificity_valid_cluster_ids_path = "/home/akubaney/projects/na_mpnn/data/datasets/specificity_dataset_v2/valid_protein_chain_cluster_ids.txt"

specificity_test_csv_path = "/home/akubaney/projects/na_mpnn/data/datasets/specificity_dataset_v2/test.csv"
specificity_test_cluster_ids_path = "/home/akubaney/projects/na_mpnn/data/datasets/specificity_dataset_v2/test_protein_chain_cluster_ids.txt"

### Valid Set

In [ ]:
specificity_valid_pdbs_output_directory = os.path.join(pdb_output_directory, "specificity_valid")
specificity_valid_csv_output_path = os.path.join(csv_output_directory, "specificity_valid.csv")

specificity_valid_df = pd.read_csv(specificity_valid_csv_path)
specificity_valid_cluster_ids = read_cluster_ids_text_file(specificity_valid_cluster_ids_path)
print("Original valid dataset size:", len(specificity_valid_df))

specificity_valid_df = get_exclusive_cluster_subset(
    specificity_valid_df, 
    "protein_chain_cluster_ids", 
    specificity_valid_cluster_ids
)
print("Valid dataset size after exclusive cluster subset:", len(specificity_valid_df))

specificity_valid_df = get_ppm_subset(specificity_valid_df)
print("Valid dataset size after PPM subset:", len(specificity_valid_df))

specificity_valid_df = get_length_subset(
    specificity_valid_df,
    max_length = 1000,
    min_length = 20
)
print("Valid dataset size after length subset:", len(specificity_valid_df))

specificity_valid_df = convert_cif_to_pdb(
    specificity_valid_df, 
    specificity_valid_pdbs_output_directory, 
    use_rna_solo = False,
    pdb_id_to_rna_solo_paths = None
)
print("Valid dataset size after CIF to PDB conversion:", len(specificity_valid_df))

get_ppm_statistics(specificity_valid_df)

specificity_valid_df.to_csv(specificity_valid_csv_output_path, index = False)
specificity_valid_df

### Test Set

In [ ]:
specificity_test_pdbs_output_directory = os.path.join(pdb_output_directory, "specificity_test")
specificity_test_csv_output_path = os.path.join(csv_output_directory, "specificity_test.csv")

specificity_test_df = pd.read_csv(specificity_test_csv_path)
specificity_test_cluster_ids = read_cluster_ids_text_file(specificity_test_cluster_ids_path)
print("Original test dataset size:", len(specificity_test_df))

specificity_test_df = get_exclusive_cluster_subset(
    specificity_test_df, 
    "protein_chain_cluster_ids", 
    specificity_test_cluster_ids
)
print("Test dataset size after exclusive cluster subset:", len(specificity_test_df))

specificity_test_df = get_ppm_subset(specificity_test_df)
print("Test dataset size after PPM subset:", len(specificity_test_df))

specificity_test_df = get_length_subset(
    specificity_test_df,
    max_length = 1000,
    min_length = 20
)
print("Test dataset size after length subset:", len(specificity_test_df))

specificity_test_df = convert_cif_to_pdb(
    specificity_test_df, 
    specificity_test_pdbs_output_directory, 
    use_rna_solo = False,
    pdb_id_to_rna_solo_paths = None
)
print("Test dataset size after CIF to PDB conversion:", len(specificity_test_df))

get_ppm_statistics(specificity_test_df)

specificity_test_df.to_csv(specificity_test_csv_output_path, index = False)
specificity_test_df